# VLM-DENTAL: Rules + Skills

Post-training, inference-only harness experiments. Use the fixed **dev** split; the official validation/test split is prohibited. This notebook never launches SFT/GRPO or changes the frozen protocol. Run in an isolated Kaggle/Colab session after your checkpoint is saved. Skills retain the existing tool workflow and explicit source references.

## 1. Isolated checkout
Apply the delivered harness patches to your repository and push them yourself first. Set REPO_REF to your reviewed harness commit. This creates a separate checkout; it does not pull over running training files.

In [ ]:
from pathlib import Path
import os, sys, subprocess, json, shlex
REPO_URL = "https://github.com/rezaxr14/VLM-DENTAL.git"
REPO_REF = "harness/skills"  # Set to the published commit/branch containing this phase.
PARENT = Path('/kaggle/working' if Path('/kaggle').exists() else '/content' if Path('/content').exists() else Path.cwd())
WORK_DIR = PARENT / 'VLM-DENTAL-HARNESS'
if not (WORK_DIR / '.git').exists():
    subprocess.run(['git','clone','--branch',REPO_REF,REPO_URL,str(WORK_DIR)],check=True)
os.chdir(WORK_DIR)
print(subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip())
assert Path('scripts/harness/run_dev.py').is_file(), 'Apply the harness phase first'


## 2. Dependencies and credentials
Prefer a fresh notebook runtime. Existing training environments are not modified. Enable installation here only in this isolated runtime. Dependency versions remain those declared by the repository.

In [ ]:
INSTALL_DEPENDENCIES = False
if INSTALL_DEPENDENCIES:
    subprocess.run([sys.executable,'-m','pip','install','-e','.[api,dev]'],check=True)
from dotenv import load_dotenv
load_dotenv()
if not os.environ.get('HF_TOKEN') and Path('/kaggle').exists():
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
    except Exception:
        print('HF_TOKEN absent; public/local checkpoints remain available. Use hf auth login for private access.')


## 3. Profile, checkpoint and fixed dev inputs
Hardware profile determines experiment identity and numerical runtime settings. Never silently substitute another checkpoint. A GRPO checkpoint should point at its step/final folder; its grpo_policy adapter is selected by the existing evaluator. No optimizer state is needed for inference.

In [ ]:
HARDWARE_PROFILE = "dual_t4"  # dual_t4 | rtx4090_qlora | rtxpro6000_48gb | workstation48_lora (alias) | tpu
PROFILES = {
    'dual_t4': dict(tag='dual_t4_fp16',precision='qlora',compute='fp16',attention='sdpa',kv='on',loader='staged',threads=1),
    'rtx4090_qlora': dict(tag='rtx4090_bf16',precision='qlora',compute='bf16',attention='auto',kv='auto',loader='standard',threads=0),
    'workstation48_lora': dict(tag='workstation48_lora_bf16',precision='bf16',compute='auto',attention='auto',kv='auto',loader='standard',threads=0),
    'rtxpro6000_48gb': dict(tag='workstation48_lora_bf16',precision='bf16',compute='auto',attention='auto',kv='auto',loader='standard',threads=0),
    'tpu': dict(tag='tpu_bf16',precision='bf16',compute='auto',attention='sdpa',kv='off',loader='standard',threads=0),
}
PROFILE = PROFILES[HARDWARE_PROFILE]
HF_REPO = 'Reza-Nadimi/vlm-dental-models'
TRACK = 'with_tools'
STAGE = 'dentex_alone'
CHECKPOINT_KIND = 'sft'  # sft | grpo
GROUP_SIZE = 4
GRPO_CHECKPOINT = f'grpo-{TRACK}-final'
LOCAL_ADAPTER = None  # Exact local checkpoint path, or download only the selected adapter below.
MODEL_ID = None  # None resolves an attached/local base model using the existing resolver.
DATA_DIR = 'data'
DEV_SPLIT = 'data/splits/dentex_dev.json'
LIMIT = 2  # Tiny dev smoke first; None = the complete fixed dev split.
SEED = 42
MAX_NEW_TOKENS = 2048
MAX_TOOL_CALLS = 50
MAX_TURNS = None
CONDITIONS = ['baseline','rules_only','rules_skills_index','rules_skills_full']
RUN_NAME = 'harness_dev_01'  # Fresh name per experiment; existing outputs are never overwritten.
OUTPUT = f'paper/generated/{RUN_NAME}_{PROFILE["tag"]}.json'


In [ ]:
from dental_agent.utils.inputs import resolve_qwen_model
if MODEL_ID is None:
    MODEL_ID, _ = resolve_qwen_model()
if LOCAL_ADAPTER is None:
    from huggingface_hub import snapshot_download
    if CHECKPOINT_KIND == 'sft':
        folder = f'sft/qwen3_5_9b_sft_{TRACK}_{STAGE}_{PROFILE["precision"]}__{PROFILE["tag"]}'
    else:
        folder = f'grpo/qwen3_5_9b_grpo_{TRACK}_k{GROUP_SIZE}_{STAGE}_{PROFILE["precision"]}__{PROFILE["tag"]}/{GRPO_CHECKPOINT}'
    patterns = [f'{folder}/adapter_config.json',f'{folder}/adapter_model*',f'{folder}/sft_run_meta.json',
                f'{folder}/checkpoint_meta.json',f'{folder}/grpo_policy/adapter_config.json',f'{folder}/grpo_policy/adapter_model*']
    cache = snapshot_download(repo_id=HF_REPO,repo_type='model',allow_patterns=patterns)
    LOCAL_ADAPTER = str(Path(cache)/folder)
assert Path(LOCAL_ADAPTER).is_dir(), LOCAL_ADAPTER
assert Path(DEV_SPLIT).is_file(), 'Use the fixed committed dev split; do not regenerate it'
print('Checkpoint:',LOCAL_ADAPTER,'Output:',OUTPUT)


## 4. Input and contract audit (no model weights loaded)
Attach your training images/data to this isolated session, or set DATA_DIR to the local training-data directory. No held-out test files are read.

In [ ]:
from dental_agent.harness.rules import load_rules, validate_implementations
from dental_agent.harness.skills import SkillLibrary
from dental_agent.harness.evaluation import load_dev
validate_implementations()
ids, images, annotations, categories = load_dev(DATA_DIR,DEV_SPLIT,LIMIT,SEED)
print('Dev episodes:',len(ids))
print(SkillLibrary().index())
print(Path('docs/harness/SKILL_CLAIMS.json').read_text())


## 5. Reproducible paired dev run
Greedy decoding, the same images, seed, geometry, budgets and checkpoint across conditions. Skills are inference-only. No training calls occur. The new runner refuses existing result paths.

In [ ]:
cmd = [sys.executable,'scripts/harness/run_dev.py','--model-id',MODEL_ID,
       '--adapter-path',LOCAL_ADAPTER,'--checkpoint-kind',CHECKPOINT_KIND,
       '--data-dir',DATA_DIR,'--dev-split',DEV_SPLIT,'--split','dev',
       '--conditions',*CONDITIONS,'--precision',PROFILE['precision'],
       '--qlora-compute-dtype',PROFILE['compute'],'--attn-implementation',PROFILE['attention'],
       '--sdpa-repeat-kv',PROFILE['kv'],'--loader-mode',PROFILE['loader'],
       '--load-cpu-threads',str(PROFILE['threads']),'--seed',str(SEED),
       '--max-new-tokens',str(MAX_NEW_TOKENS),'--max-tool-calls',str(MAX_TOOL_CALLS),'--output',OUTPUT]
if LIMIT is not None: cmd += ['--limit',str(LIMIT)]
if MAX_TURNS is not None: cmd += ['--max-turns',str(MAX_TURNS)]
print(shlex.join(cmd))
subprocess.run(cmd,check=True)


## 6. Report and skill-load audit
Pooled exact F1 and invented-findings rate use shared matching. Undefined rates are null, never invented as zero. Inspect format failures alongside clinical metrics.

In [ ]:
report = json.loads(Path(OUTPUT).read_text())
for name,value in report['conditions'].items():
    print(name,json.dumps(value.get('summary'),indent=2))
    print('Skill events:',[(r['image_id'],r.get('harness_events')) for r in value['records']])
print('Paired differences:',json.dumps(report['comparisons'],indent=2))


## 7. Protocol amendment — review only
The amendment below is a proposal. This notebook does not apply it or edit the frozen protocol.

In [ ]:
print(Path('docs/harness/proposals/frozen_protocol_amendment.patch').read_text())


## 8. Export
Download the result and the claim/protocol reports. No automatic Hub upload or git push.

In [ ]:
import zipfile
bundle = Path(OUTPUT).with_suffix('.zip')
with zipfile.ZipFile(bundle,'w',zipfile.ZIP_DEFLATED) as z:
    for p in [Path(OUTPUT),Path('docs/harness/SKILL_CLAIMS.json'),Path('docs/harness/proposals/frozen_protocol_amendment.patch')]:
        z.write(p,arcname=p.name)
print('Saved:',bundle.resolve())
try:
    from google.colab import files
    files.download(str(bundle))
except ImportError:
    from IPython.display import FileLink, display
    display(FileLink(str(bundle)))
